FastAPI

Triton Inference Server
-   собирает батчи
-   умеет раюотать с registry моделей

Сейчас называется Nvidia Dynamo

### Serving
Serving = запуск модели в режиме сервера: вместо ad-hoc запусков, процесс доступен 24/7 и клиенты отправляют запросы на выполнение

Есть еще batch serving - когда по расписанию

Streaming serving - анализ постоянного сигнала

Так как это production режим, вощникает ряд требований:
- быстрота ответа latency
- валидация входа выхода
- универсальность



### Nvidia Triton Inference
Triton = инструмент сервинга, не привязанный к конкретному бекенду

Cейчас называется Dynamo. 

<img src="img/triton1.jpg" width=400>

### Docker
Docker = система управления контейнерами. Контейнер = процесс, изолированный с преднастроенным окружением (библиотеки)

Почему полезна контейнеризвация:
- Изоляция окружения<br>нет конфликта версий библиотек
- Переносимость (works everywhere)<br>одинаково ставится на разные версии Linux
- Быстрое масштабирование<br>дополнительный экземпляр процесса запускается за секунды
- Изоляция ресурсов<br>контролирование использование
- Повторяемость (reproducibility)<br>экономия на настройке
- Упрощённая доставка (CI/CD)<br>экономия на настройке при развертывании
- Безопасность<br>ограничиваются права, системных вызорвов
- Изоляция сети<br>снаружи видны как отдельные машины

Что конкретно изолируется в рамках контейнера?
- с помощью __namespaces__ выделяется: свое дерево процессов PID, файловая система FS, доменное имя, сетевой менеджемент (iptables etc)
- с помощью __cgroups__ выделяются свои лимиты на ресурсы (cpu, ram, i/o, network)
- с помощью OverlayFS создается layered файловая системы
- используются инструменты (например, sescomp), чтобы дропунть потенциально опасный функционал

*OverlayedFS - "слоеная" файловая система, когда есть read-only базовый слой, а все модификации наслаиваются в новых слоях. В базовом слое либо голая операционная система (точнее пользовательская ее часть User-space Linux, ядро)

Ядро Linux = драйверы, системные вызовы. Пользоватлеьская часть = shell, более выскоуровневые библиотеки

Можно запускать не только фоне, но и интерактивный процесс (флаг -i -t создает). Можно подключаться к работающему конейтнеру

Что происходит при docker run:
- идет обращение к оркестрирующему процессу dockerd, который создает в своей таблице запись про новый контейнер
- форкается новый процесc командой clone (сразу с изоляцией - с созданием новых пространств имен)
- собирается новая файловая система (mount overlay) и её каталог делается корневым (chroot)
- настраиваются лимиты (с помощью namespaces, cgroups, seccomp)
- запуск целевой команды через execve 

Docker-compose - обертка над командой Docker. Работает не с отдельными Docker-скриптами, а с файлом конфигурации (yml). Удобно для разворачивания FastAPI + Triton

В конфигурации указываем ключевые поля: services, image, volumes, ports, 

CI/CD регламентирует релиз новых версий приложения
Традиционно функционал CI/CD есть в GitLab
по событию (push, manual run) => сборка + тест + deploy на сервер(а)
Ansible: скрипт для менеджемнта парка машин
Kubernetes: для оркестрации